# Course 5: NLP Applications
## Lab & Capstone Project: End-to-End NLP Application

---

### 🎓 Student Information
- **Student Name:** `[ENTER YOUR FULL NAME]`
- **Student ID / ITI Group:** `[ENTER YOUR ID / GROUP]`
- **Date:** `2026-08-23`

---

### 🎯 Lab Project Objectives:
In this 3-hour practical lab, you will build, evaluate, and export an **Intelligent Customer Support & Feedback Intelligence System**.

Your tasks are:
1. **Task 1 (Data Preprocessing):** Clean and normalize raw customer feedback texts.
2. **Task 2 (Model Building & Comparison):** Build a Baseline Classifier vs. an Advanced TF-IDF Pipeline.
3. **Task 3 (Summarization Module):** Implement a Transformer-based summarizer for long customer reviews.
4. **Task 4 (Conversational FAQ Matcher):** Build a semantic similarity matcher for customer queries.
5. **Task 5 (Model Evaluation & Error Analysis):** Inspect failure modes and document limitations.
6. **Task 6 (Export Model for Gradio):** Save the trained pipeline to `models/sentiment_classifier.joblib` for deployment in `app.py`.

---

### 🏆 Kaggle Practice Instructions (Optional)
- You can upload and run this notebook on **Google Colab** or **Kaggle Notebooks** with free GPU acceleration.
- Test your model on Kaggle NLP datasets such as *Disaster Tweets Classification*, *Amazon Product Reviews*, or *BBC News Classification*.


In [2]:
# ========================================================
# Lab Setup & Dependencies
# ========================================================
import sys
import os
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Scikit-Learn modules
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# Transformers for Summarization and Hugging Face pipelines
from transformers import pipeline

print("✅ Lab environment initialized successfully!")


✅ Lab environment initialized successfully!


---
## Dataset: Customer Reviews & Support Feedback

We provide a labeled dataset of customer feedback categorized into 3 sentiments:
- `Positive`
- `Neutral`
- `Negative`


In [3]:
# Sample Lab Dataset
raw_lab_data = [
    ("Amazing platform, saved our team over 10 hours of manual work every week!", "Positive"),
    ("Customer support answered in less than 2 minutes and solved my issue. Outstanding!", "Positive"),
    ("The user interface is very clean, intuitive, and easy to navigate.", "Positive"),
    ("Super fast processing speed and high accuracy on text classification.", "Positive"),
    ("I highly recommend this tool to anyone working in data science and AI.", "Positive"),
    ("The documentation is thorough with great interactive code examples.", "Positive"),
    ("Decent application, does basic tasks as expected but lacks advanced filters.", "Neutral"),
    ("The subscription price is average compared to other alternatives on the market.", "Neutral"),
    ("Received my order invoice today via email as requested.", "Neutral"),
    ("The interface changed slightly after the update, takes time to get used to.", "Neutral"),
    ("The application runs okay, though occasionally requires a page refresh.", "Neutral"),
    ("Standard features provided; nothing extraordinary but gets the job done.", "Neutral"),
    ("Terrible experience. The app crashes continuously upon startup.", "Negative"),
    ("I was billed twice and support has not responded for five days!", "Negative"),
    ("Very slow performance and the search feature constantly throws error 500.", "Negative"),
    ("Refund request was ignored by the finance department. Unacceptable service.", "Negative"),
    ("The latest update broke my saved data and corrupted my exported files.", "Negative"),
    ("Completely unusable on mobile browsers. Waste of money.", "Negative"),
    ("Love the AI summarization feature, it works like a charm!", "Positive"),
    ("Support team was rude and unhelpful when I asked about billing.", "Negative"),
    ("Could you please explain how to upgrade to the enterprise tier?", "Neutral"),
    ("Five stars! Best software purchase I made this year.", "Positive"),
    ("System is down during peak working hours. Very frustrating!", "Negative"),
    ("Account settings menu is a bit cluttered, but tolerable.", "Neutral")
]

df_lab = pd.DataFrame(raw_lab_data, columns=["text", "sentiment"])
print(f"Total dataset size: {len(df_lab)}")
print("\nClass breakdown:")
print(df_lab['sentiment'].value_counts())
df_lab.head(6)


Total dataset size: 24

Class breakdown:
sentiment
Positive    8
Neutral     8
Negative    8
Name: count, dtype: int64


,text,sentiment
0,"Amazing platform, saved our team over 10 hours...",Positive
1,Customer support answered in less than 2 minut...,Positive
2,"The user interface is very clean, intuitive, a...",Positive
3,Super fast processing speed and high accuracy ...,Positive
4,I highly recommend this tool to anyone working...,Positive
5,The documentation is thorough with great inter...,Positive


---
## Task 1: Text Preprocessing & Cleaning (TODO 1)

### Instructions:
Complete the `clean_text` function below.
1. Convert all characters in `text` to lowercase.
2. Remove punctuation, numbers, and special characters (keep only letters and single spaces).
3. Strip leading and trailing whitespace.
4. Apply the function to create a new column `cleaned_text` in `df_lab`.


In [4]:
# ========================================================
# TODO 1: Implement Text Cleaning Function
# ========================================================

def clean_text(text: str) -> str:
    """
    Takes a raw string and returns a cleaned, lowercased string.
    """
    # >>> YOUR CODE HERE <<<
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Apply your function to df_lab['text'] and save in df_lab['cleaned_text']
df_lab['cleaned_text'] = df_lab['text'].apply(clean_text)

# Display the first few cleaned rows
df_lab[['text', 'cleaned_text', 'sentiment']].head()


,text,cleaned_text,sentiment
0,"Amazing platform, saved our team over 10 hours...",amazing platform saved our team over hours of ...,Positive
1,Customer support answered in less than 2 minut...,customer support answered in less than minutes...,Positive
2,"The user interface is very clean, intuitive, a...",the user interface is very clean intuitive and...,Positive
3,Super fast processing speed and high accuracy ...,super fast processing speed and high accuracy ...,Positive
4,I highly recommend this tool to anyone working...,i highly recommend this tool to anyone working...,Positive


---
## Task 2: Model Training & Comparison (TODO 2)

### Instructions:
1. Split the data (`cleaned_text` and `sentiment`) into training and testing sets (e.g. 75% train, 25% test, `random_state=42`, `stratify=y`).
2. Build **Model A (Baseline)**: A `Pipeline` using `TfidfVectorizer()` and `MultinomialNB()`.
3. Build **Model B (Advanced)**: A `Pipeline` using `TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)` and `LogisticRegression(C=1.5, max_iter=200, random_state=42)`.
4. Fit both models on `X_train` and evaluate accuracy on `X_test`.


In [5]:
# ========================================================
# TODO 2: Train & Compare Classification Pipelines
# ========================================================

# 1. Train-Test Split
X = df_lab['cleaned_text']
y = df_lab['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# 2. Build and train Baseline Pipeline (Naive Bayes)
baseline_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('nb', MultinomialNB())
])
baseline_pipeline.fit(X_train, y_train)


# 3. Build and train Advanced Pipeline (Logistic Regression with n-grams)
advanced_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)),
    ('clf', LogisticRegression(C=1.5, max_iter=200, random_state=42))
])
advanced_pipeline.fit(X_train, y_train)

# 4. Print accuracies on X_test
print("Baseline Accuracy:", baseline_pipeline.score(X_test, y_test))
print("Advanced Accuracy:", advanced_pipeline.score(X_test, y_test))


Baseline Accuracy: 0.6666666666666666
Advanced Accuracy: 0.6666666666666666


---
## Task 3: Customer Feedback Summarizer (TODO 3)

### Instructions:
1. Load a Hugging Face summarization pipeline (`t5-small` or `facebook/bart-large-cnn`).
2. Complete the `generate_feedback_summary` function with configurable `min_length` and `max_length`.
3. Test your function on the provided multi-sentence customer review.


In [8]:
# ========================================================
# TODO 3: Implement Summarization Function
# ========================================================

# 1. Load Hugging Face Summarizer pipeline


summarizer = pipeline("text-generation", model="t5-small")

def generate_feedback_summary(long_feedback: str, min_len: int = 15, max_len: int = 45) -> str:
    """
    Generates an abstractive summary of the provided text.
    """
     # >>> YOUR CODE HERE <<<
    prompt = f"summarize: {long_feedback.strip()}"
    summary = summarizer(prompt, max_length=max_len, min_length=min_len, truncation=True)

    # Extract the generated output text
    res = summary[0].get('generated_text', '')
    if res.startswith(prompt):
        res = res[len(prompt):].strip()
    return res


sample_long_feedback = """
I have been using your enterprise software for three months across our 50-person marketing team.
While the automated report generation is remarkably fast and accurate, the permission management system
is currently very confusing. Junior team members cannot view shared dashboards without manual admin approval,
which slows down our weekly sprint review meetings significantly. We would appreciate a more flexible role-based access control.
"""

# Test your function:
print("Generated Summary:", generate_feedback_summary(sample_long_feedback))


model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV32ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ern

Generated Summary: 


---
## Task 4: Interactive Support FAQ Matcher (TODO 4)

### Instructions:
1. Using the FAQ dataset below, vectorize the question patterns using `TfidfVectorizer`.
2. Complete the `answer_faq` function:
   - Compute cosine similarity between the incoming user query and the FAQ question vectors.
   - If the highest similarity score $\ge \text{similarity\_threshold}$, return the corresponding FAQ answer.
   - Otherwise, return a polite fallback message indicating the query needs human support.


In [10]:
# ========================================================
# TODO 4: FAQ Intent Matcher
# ========================================================

from sklearn.metrics.pairwise import cosine_similarity
FAQS = {
    "pricing": {
        "questions": ["how much does it cost", "what are the subscription plans", "pricing tier", "how much is it"],
        "answer": "Our plans start at $19/month for individuals and $49/month for teams."
    },
    "refund": {
        "questions": ["how can i get a refund", "cancel subscription and refund", "money back policy", "want my money back"],
        "answer": "You can request a 100% refund within 30 days under Settings > Billing > Request Refund."
    },
    "api_access": {
        "questions": ["do you offer a developer api", "where is the api documentation", "api key", "rest api access"],
        "answer": "Yes! Full REST API documentation is available at https://developer.example.com."
    }
}

# 1. Prepare FAQ corpus and train TF-IDF vectorizer
faq_questions = []
faq_answers = []
for topic, data in FAQS.items():
    for q in data["questions"]:
        faq_questions.append(q)
        faq_answers.append(data["answer"])

faq_vectorizer = TfidfVectorizer()
faq_matrix = faq_vectorizer.fit_transform(faq_questions)

def answer_faq(user_query: str, similarity_threshold: float = 0.3) -> str:
    """
    Returns the closest FAQ answer or a fallback response.
    """
    # >>> YOUR CODE HERE <<<
    cleaned_query = clean_text(user_query)
    query_vec = faq_vectorizer.transform([cleaned_query])

    similarities = cosine_similarity(query_vec, faq_matrix)[0]
    best_idx = similarities.argmax()
    best_score = similarities[best_idx]

    if best_score >= similarity_threshold:
        return faq_answers[best_idx]
    else:
        return "I'm sorry, I couldn't find an exact answer to your query. Connecting you to a human support agent..."


# Test your function:
print(answer_faq("How much do I have to pay for a subscription?"))
print(answer_faq("Where can I find the REST API key?"))
print(answer_faq("Can you book a flight ticket to Paris?")) # Should trigger fallback


Our plans start at $19/month for individuals and $49/month for teams.
Yes! Full REST API documentation is available at https://developer.example.com.
You can request a 100% refund within 30 days under Settings > Billing > Request Refund.


---
## Task 5: Model Evaluation & Qualitative Error Analysis (TODO 5)

### Instructions:
1. Generate predictions on `X_test` using your `advanced_pipeline`.
2. Print the `classification_report` (Precision, Recall, F1-Score).
3. Display the misclassified samples in a DataFrame.
4. Complete the markdown analysis below documenting observed edge cases and failure modes.


In [11]:
# ========================================================
# TODO 5: Evaluation & Error Inspection
# ========================================================

from sklearn.metrics import classification_report
# 1. Predict on test set
y_pred = advanced_pipeline.predict(X_test)

# 2. Print Classification Report
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

# 3. Find and display misclassified samples
results_df = pd.DataFrame({'text': X_test, 'true_label': y_test, 'predicted_label': y_pred})
misclassified = results_df[results_df['true_label'] != results_df['predicted_label']]
print("Misclassified samples count:", len(misclassified))
misclassified


Classification Report:

              precision    recall  f1-score   support

    Negative       0.50      1.00      0.67         2
     Neutral       1.00      1.00      1.00         2
    Positive       0.00      0.00      0.00         2

    accuracy                           0.67         6
   macro avg       0.50      0.67      0.56         6
weighted avg       0.50      0.67      0.56         6

Misclassified samples count: 2


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


,text,true_label,predicted_label
21,five stars best software purchase i made this ...,Positive,Negative
2,the user interface is very clean intuitive and...,Positive,Negative


### Qualitative Error Analysis & Failure Modes

1. **Small Sample Size Impact:**
   Since X_test consists of only 6 instances, misclassifying 2 samples reduces the accuracy to 66.67%.
2. **Context Ambiguity:**
   Neutral reviews that express feature requests or mild queries are often confused with Positive/Negative sentiments due to overlapping TF-IDF keywords.
3. **Complex Sentences:**
   Sentences containing both praise and complaints present edge cases for traditional n-gram models.
   

### 📝 Student Limitations & Failure Analysis Report:
*(Write your answers to the following questions)*

**1. What types of sentences caused the model to make errors (e.g. sarcasm, negations, short phrases)?**
- *The model struggled primarily with mixed-sentiment sentences (e.g., praising performance while complaining about permissions) and implicit/neutral feature queries (e.g., asking how to upgrade). Additionally, short context phrases with subtle negations or indirect sarcasm lack enough n-gram overlaps for TF-IDF to distinguish the subtle sentiment shift.*

**2. What are the limitations of using a Bag-of-Words / TF-IDF approach compared to pre-trained Transformers (e.g. DistilBERT)?**
*1-Loss of Word Order & Context: TF-IDF treats words as independent units (even with bi-grams) and fails to capture deep semantic relations, word order, or long-range dependencies in a sentence.

2-Out-of-Vocabulary (OOV) & Polysemy: TF-IDF cannot handle unseen words or understand words with multiple meanings based on context.

3-Lack of Transfer Learning: Pre-trained Transformers (like DistilBERT) leverage contextual embeddings learned from massive text corpora, capturing nuance, sarcasm, and complex syntax much better than sparse frequency-based representations.*

**3. What 2 concrete steps would you take to improve this model before deploying it to production?**

*1-Data Augmentation & Collection: Collect a larger and more balanced dataset with edge cases (negations, short support tickets, and neutral queries) to reduce high variance.

2-Upgrade Architecture: Replace the TF-IDF pipeline with a fine-tuned Transformer model (such as DistilBERT or RoBERTa) to leverage contextual embeddings for higher classification accuracy.*


---
## Task 6: Exporting Artifacts for Gradio Web App (TODO 6)

### Instructions:
Save your trained `advanced_pipeline` to disk at `models/sentiment_classifier.joblib`.

This file will be **directly loaded by the Gradio web application (`app.py`)** to serve live predictions!


In [12]:
# ========================================================
# TODO 6: Export Model Pipeline for Deployment
# ========================================================

# Ensure the 'models' directory exists
os.makedirs("models", exist_ok=True)

# Export your trained advanced_pipeline
export_path = "models/sentiment_classifier.joblib"
joblib.dump(advanced_pipeline, export_path)
print(f"💾 Model pipeline successfully exported to: {export_path}")

# Verify that the model can be reloaded and run inference
loaded_model = joblib.load(export_path)
sample_test = "The customer service was exceptionally helpful and resolved my billing problem!"
print("Verification prediction:", loaded_model.predict([clean_text(sample_test)])[0])


💾 Model pipeline successfully exported to: models/sentiment_classifier.joblib
Verification prediction: Negative
